In [ ]:
import os
import sys
import time
import json
import random
import argparse
from pathlib import Path

import torch

# Repo setup
repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from argparse import Namespace
from pathlib import Path
import torch

from revlm.config_utils import configure_args
from revlm.run.edit import run_edit

editor_name = "ike_chain"


In [ ]:

# Project root
project_root = Path(repo_root)

# Build test paths from args (no hardcoding)
args = Namespace(
    config="revlm/config/config.yaml",
    editor=editor_name,
    model_name="qwen3_4b",
    dataset_name="aokvqa",
    task="mc",
    batch_size=1,
    split="all",
    rationale=False,
    cot=False,
    subsample=500,
    subsample_edits=100,
    overwrite=True,
)

# Derive result prefix from args
res_prefix = project_root / "results" / "test" / f"editor_{args.editor}" / args.model_name / args.dataset_name
res_prefix.mkdir(parents=True, exist_ok=True)
(res_prefix / "pred_postedit").mkdir(parents=True, exist_ok=True)

# Force all outputs into the test prefix (overwrite allowed)
args.task_dir = str(res_prefix)
args.edit_dir = str(res_prefix)
args.pred_dir = str(res_prefix)
args.pred_path = str(res_prefix / f"pred_mc.json")
args.pred_postedit_dir = str(res_prefix / f"pred_postedit")

args.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
args.suffix = "_cot" if args.rationale and args.cot else ("_rationale" if args.rationale else "")

config = configure_args(args, config_path=args.config)
config.subsample = args.subsample
config.subsample_edits = args.subsample_edits
config.rationale = args.rationale
config.cot = args.cot
config.pred_path = args.pred_path
config.overwrite = args.overwrite
config.task_dir = args.task_dir
config.pred_dir = args.pred_dir
config.pred_postedit_dir = args.pred_postedit_dir
config.edit_dir = args.edit_dir

config.plot_k_dist = True 
run_edit(config, sequential=True, eval_every=20)